# Classificação de Dados com MobileNetV3 (timm)

Este notebook demonstra como utilizar o modelo `timm/mobilenetv3_small_100.lamb_in1k` da biblioteca `timm` para classificar imagens localizadas na pasta `images`.

In [2]:
import torch
import timm
from PIL import Image
from torchvision import transforms
import pathlib
import urllib.request
import json

# Nome exato conforme solicitado (formatado para o timm/hf_hub)
MODEL_NAME = 'hf_hub:timm/mobilenetv3_small_100.lamb_in1k'
IMAGE_DIR = pathlib.Path('images') # Pasta de imagens na raiz do projeto

if not IMAGE_DIR.exists():
    IMAGE_DIR = pathlib.Path('../images')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: cpu


### 1. Inicializando o Modelo e Transformações

In [3]:
# Cria o modelo e carrega pesos pré-treinados
model = timm.create_model(MODEL_NAME, pretrained=True)
model = model.to(device)
model.eval()

# Obtém as transformações recomendadas pelo próprio modelo
data_config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_config, is_training=False)

print(f"Configuração do modelo: {data_config}")

Configuração do modelo: {'input_size': [3, 224, 224], 'interpolation': 'bicubic', 'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225], 'crop_pct': 0.875, 'crop_mode': 'center'}


### 2. Obtendo as Etiquetas do ImageNet
Carregamos as etiquetas do ImageNet-1k para interpretar os índices de predição.

In [4]:
def get_imagenet_labels():
    url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
    with urllib.request.urlopen(url) as response:
        # Remove espaçamento e aspas potenciais
        labels = [line.decode("utf-8").strip().replace("'", "").replace(",", "") for line in response.readlines()]
    return labels

labels = get_imagenet_labels()

### 3. Classificando Imagens da pasta `images`

In [5]:
image_paths = list(IMAGE_DIR.glob("*"))
image_paths = [p for p in image_paths if p.suffix.lower() in [".jpg", ".png", ".jpeg"]]

if not image_paths:
    print(f"Nenhuma imagem encontrada em: {IMAGE_DIR.absolute()}")
else:
    for img_path in image_paths:
        img = Image.open(img_path).convert("RGB")
        
        # Processamento e Inferência
        input_tensor = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            output = model(input_tensor)
            
        probabilities = torch.nn.functional.softmax(output[0], dim=0)
        top5_prob, top5_idx = torch.topk(probabilities, 5)
        
        print(f"--- {img_path.name} ---")
        for i in range(5):
            print(f"{i+1}. {labels[top5_idx[i].item()]}: {top5_prob[i].item():.2%}")
        print()

--- bird.jpg ---
1. bulbul: 14.63%
2. chickadee: 11.52%
3. junco: 9.89%
4. indigo bunting: 4.72%
5. jay: 4.67%

--- cats_dog.png ---
1. Rhodesian ridgeback: 12.38%
2. basenji: 12.06%
3. dingo: 9.30%
4. redbone: 6.86%
5. kelpie: 3.58%

--- kitchen.png ---
1. plate rack: 11.20%
2. dining table: 10.31%
3. altar: 8.53%
4. microwave: 5.08%
5. china cabinet: 4.85%

--- pizza.png ---
1. pizza: 96.34%
2. French loaf: 0.35%
3. pretzel: 0.22%
4. hot pot: 0.17%
5. bakery: 0.14%

